# Aligned Classification — LLM-Judge Prompt Optimiser

Iteratively aligns a judge prompt with human-expert golden labels from
`classification_golden_test_cases.xlsx`.

**Flow**
1. Start with `prompt_0` — a judge that evaluates the `narrative` field generated by the intent-classification agent.
2. For each round, send every test-case row to the webhook with `{"input": filled_prompt}`.
3. Parse the JSON response; compare predicted labels against golden labels.
4. If **all** fields have mismatch rate ≤ 30 % → stop.
5. Otherwise feed mismatched rows into `OptPrompt` to obtain an improved prompt, then repeat.
6. Stop after **5 rounds** at most.

### What is evaluated?
The classification agent outputs two fields:
- `route_to` — deterministic; evaluated by exact match against `expected_route`.
- `result_narrative` — free-text; evaluated here by the LLM judge.

**Score field:** `narrative`  →  golden column: `rating_narrative`  
**Allowed values:** `"good"` / `"acceptable"` / `"invalid"`

In [1]:
import os
import re
import json
import time
import requests
import pandas as pd
from tqdm.notebook import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
N8N_BASE_URL       = "https://alphamakeathon-automation.arisetech.dev"
WEBHOOK_PATH       = "9d6049fc-77c8-43e9-b71e-f734506f4f9d"   # ← fill in
USE_TEST_URL       = False   # True → /webhook-test/…  |  False → /webhook/…

GOLDEN_FILE        = "golden_test_cases/classification_golden_test_cases.xlsx"
RESULT_DIR         = "classification_test_result"

MAX_ROUNDS         = 5
MISMATCH_THRESHOLD = 0.30    # stop when ALL fields are within this rate
TIMEOUT            = 600      # seconds per request (10 minutes)
RETRIES            = 2
DELAY_BETWEEN      = 0.5     # seconds between rows

SCORE_FIELDS = ["narrative"]


def golden_col(f: str) -> str:
    """Map a score-field name to its golden-label column in the DataFrame."""
    return f"rating_{f}"


def get_webhook_url() -> str:
    prefix = "webhook-test" if USE_TEST_URL else "webhook"
    return f"{N8N_BASE_URL}/{prefix}/{WEBHOOK_PATH}"


print("Webhook URL:", get_webhook_url())

Webhook URL: https://alphamakeathon-automation.arisetech.dev/webhook/9d6049fc-77c8-43e9-b71e-f734506f4f9d


## OptPrompt

Meta-prompt sent to the webhook when mismatches exceed the threshold.
Placeholders: **`{oldPrompt}`**, **`{mismatchResults}`**.
The LLM must return an *UpdatePrompt* whose own placeholders cover all
conversation-context fields and the generated narrative field listed below.

In [2]:
OptPrompt = (
    "You are an expert prompt engineer specialising in LLM-as-a-judge systems "
    "for evaluating AI-generated routing narratives at a Thai commercial bank.\n\n"
    "Your task is to improve the judge prompt below so it better aligns with "
    "human-expert (golden) evaluations.\n\n"
    "## Current Prompt\n"
    "{oldPrompt}\n\n"
    "## Mismatch Cases\n"
    "The following rows show where the current prompt disagreed with golden labels. "
    "Each entry includes the conversation context, the generated narrative, "
    "the golden label, and the predicted label:\n"
    "{mismatchResults}\n\n"
    "## Your Task\n"
    "1. Analyse the mismatch patterns.\n"
    "2. Identify what the current prompt misinterprets.\n"
    "3. Rewrite the prompt to fix those issues while preserving correct behaviours.\n\n"
    "## Requirements for the output prompt\n"
    "- MUST keep exactly these placeholders (with single curly braces):\n"
    "  {userMessage}, {LastAImessage}, {PrevAIagent}, {narrative},\n"
    "  {expected_route}, {reason}, {result_narrative}\n"
    "- MUST instruct the LLM to return ONLY a JSON object with this field and allowed values:\n"
    '    { "narrative": "good"|"acceptable"|"invalid" }\n\n'
    "Return ONLY the improved prompt text — no preamble, no explanation."
)

print("OptPrompt defined:", len(OptPrompt), "chars")

OptPrompt defined: 1119 chars


## Initial Judge Prompt — `prompt_0`

Evaluates the quality of the `result_narrative` generated by the classification agent.
Placeholders: **`{userMessage}`**, **`{LastAImessage}`**, **`{PrevAIagent}`**,
**`{narrative}`**, **`{expected_route}`**, **`{reason}`**, **`{result_narrative}`**.

Returns JSON with `narrative`, valued **`"good"`** / **`"acceptable"`** / **`"invalid"`**.

**Rating criteria:**
- `good` — Accurately captures the customer's **specific** intent with relevant details;
  clearly explains why the conversation is routed to the correct destination.
- `acceptable` — Correct routing direction but too **generic or formulaic**;
  does not synthesise the customer's specific situation.
- `invalid` — **Factually wrong**: misrepresents what the customer said,
  attributes actions the customer has not taken, or contradicts the correct routing.

In [3]:
prompt_0 = (
    "You are a Quality Assurance (QA) expert evaluating AI-generated routing narratives "
    "for a Thai bank chatbot system.\n\n"
    "The chatbot classifies customer messages and routes them to the appropriate agent:\n"
    '- "advisor"   : customer needs a personalised debt solution or restructuring plan.\n'
    '- "education" : customer asks a general question about products or programmes.\n'
    '- "summary"   : staff has requested a case summary.\n'
    '- "unknown"   : intent is unclear or off-topic.\n\n'
    "Along with the routing decision the system generates a brief narrative that describes "
    "the conversation context and the customer's intent. Your task is to evaluate whether "
    "this generated narrative is accurate and useful.\n\n"
    "## Conversation Context\n\n"
    "Customer's Latest Message:\n"
    "{userMessage}\n\n"
    "Previous AI Message:\n"
    "{LastAImessage}\n\n"
    "Previous Agent Type:\n"
    "{PrevAIagent}\n\n"
    "Running Conversation Narrative (from prior turns):\n"
    "{narrative}\n\n"
    "## Classification Result\n\n"
    "Correct Route:\n"
    "{expected_route}\n\n"
    "Routing Reason:\n"
    "{reason}\n\n"
    "Generated Narrative to Evaluate:\n"
    "{result_narrative}\n\n"
    "## Evaluation Criteria\n\n"
    "### narrative — Quality of the generated routing narrative\n\n"
    '- "good"       : The narrative accurately and specifically describes the customer\'s '
    "key intent and the conversation state. It captures relevant details from the customer's "
    "message (e.g. specific financial situation, what they are requesting) and clearly "
    "explains why the conversation routes to the correct destination. A good narrative "
    "synthesises the situation rather than merely restating the customer's words or "
    "using a generic template phrase.\n\n"
    '- "acceptable" : The narrative is directionally correct (supports the right routing) '
    "but is too generic or formulaic. It may use template phrases such as "
    '"Customer\'s latest message indicates intent related to {route}" without adding specific '
    "context about *what* the customer actually said or *why* the routing applies. "
    "The core intent is captured but the narrative does not synthesise the customer's "
    "specific situation.\n\n"
    '- "invalid"    : The narrative is factually wrong. Examples: it states the customer '
    "accepted an offer when they have not, attributes actions or statements to the customer "
    "that are not present in the conversation, contradicts the correct routing, "
    "or fabricates context that does not exist.\n\n"
    "## Output Instructions\n"
    "Return ONLY a JSON object — no markdown, no explanation:\n"
    '{\n'
    '  "narrative": "good" | "acceptable" | "invalid"\n'
    '}'
)

print("prompt_0 defined:", len(prompt_0), "chars")

prompt_0 defined: 2403 chars


## Helper Functions

In [4]:
_PROMPT_FIELDS = [
    "userMessage", "LastAImessage", "PrevAIagent", "narrative",
    "expected_route", "reason", "result_narrative",
]


def fill_prompt(template: str, row: pd.Series) -> str:
    filled = template
    for col in _PROMPT_FIELDS:
        filled = filled.replace(f"{{{col}}}", str(row.get(col, "")))
    return filled


def fill_opt_prompt(template: str, oldPrompt: str, mismatchResults: str) -> str:
    return (
        template
        .replace("{oldPrompt}",       str(oldPrompt))
        .replace("{mismatchResults}", str(mismatchResults))
    )


def _call_raw(payload: dict, timeout: int = TIMEOUT, retries: int = RETRIES):
    url = get_webhook_url()
    last_exc: Exception | None = None
    for attempt in range(retries + 1):
        try:
            resp = requests.post(url, json=payload, timeout=timeout)
            resp.raise_for_status()
            return resp.json()
        except requests.exceptions.RequestException as exc:
            last_exc = exc
            if attempt < retries:
                time.sleep(1.5 * (attempt + 1))
    raise last_exc


def call_webhook(prompt: str):
    return _call_raw({"input": prompt})


def get_response_text(resp) -> str:
    if isinstance(resp, str):
        return resp
    if isinstance(resp, dict):
        for key in ("output", "text", "result", "response", "content", "message"):
            if key in resp and resp[key] is not None:
                return str(resp[key])
        return json.dumps(resp, ensure_ascii=False)
    return str(resp)


def parse_json_response(resp) -> dict:
    text = get_response_text(resp)
    # 1. markdown code block
    m = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    # 2. whole text as JSON
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # 3. first JSON object found
    m = re.search(r"(\{[\s\S]*?\})", text)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    return {}


def extract_prompt_text(resp) -> str:
    text = get_response_text(resp)
    m = re.search(r"```(?:\w+)?\s*([\s\S]*?)\s*```", text)
    if m:
        return m.group(1).strip()
    return text.strip()


print("Helpers ready.")

Helpers ready.


## Load Golden Test Cases

In [5]:
df_golden = pd.read_excel(GOLDEN_FILE)

# Normalise labels to lowercase so they match the judge's output values
for f in SCORE_FIELDS:
    df_golden[golden_col(f)] = df_golden[golden_col(f)].str.strip().str.lower()

print(f"Loaded {len(df_golden)} golden test cases")
print("Columns:", df_golden.columns.tolist())
print("\nLabel distributions:")
for f in SCORE_FIELDS:
    print(f"  {golden_col(f)}: {df_golden[golden_col(f)].value_counts().to_dict()}")
print("\nexpected_route distribution:", df_golden["expected_route"].value_counts().to_dict())
df_golden.head(3)

Loaded 30 golden test cases
Columns: ['testId', 'userMessage', 'LastAImessage', 'PrevAIagent', 'narrative', 'expected_route', 'reason', 'result_narrative', 'rating_narrative', 'reviewed_rating_narrative', 'rating_review_status', 'review_comment']

Label distributions:
  rating_narrative: {'good': 20, 'acceptable': 6, 'invalid': 4}

expected_route distribution: {'advisor': 12, 'education': 8, 'unknown': 6, 'summary': 4}


,testId,userMessage,LastAImessage,PrevAIagent,narrative,expected_route,reason,result_narrative,rating_narrative,reviewed_rating_narrative,rating_review_status,review_comment
0,TC-0001,ตอนนี้โบนัสถูกเลื่อน เลยอยากถามว่าอยากลดค่างวด...,เสนอแผนการปรับโครงสร้างสินเชื่อ ด้วยแผนงานและข...,ADVISOR,Customer raises a personal debt-management sit...,advisor,Customer requests a personalized debt solution...,Customer's latest message indicates intent rel...,acceptable,acceptable,unchanged,"Acceptable: route is correct, but the narrativ..."
1,BAD,ผมไม่ได้อยากหนีหนี้นะครับ แต่แม่เข้าโรงพยาบาล ...,เสนอแผนการปรับโครงสร้างสินเชื่อ ด้วยแผนงานและข...,ADVISOR,Customer raises a personal debt-management sit...,advisor,Customer requests a personalized debt solution...,Customer has already accepted the restructurin...,invalid,invalid,unchanged,Invalid: the customer asks for a lower-payment...
2,TC-0018,แผนที่ให้มาดูดีนะคะ แต่พอดีเพิ่งเปลี่ยนงาน เลย...,เสนอแผนการปรับโครงสร้างสินเชื่อ ด้วยแผนงานและข...,ADVISOR,Customer raises a personal debt-management sit...,advisor,Customer requests a personalized debt solution...,Customer's latest message indicates intent rel...,good,good,unchanged,Good under relaxed threshold: the narrative ro...


## Optimisation Loop

For each round:
1. Fill all conversation-context and generated-narrative placeholders in the current judge prompt.
2. POST `{"input": filled_prompt}` to the webhook; collect responses.
3. Parse each response string → dict with `re` / `json`.
4. Save `test_result_prompt_N.xlsx`.
5. Compute per-field mismatch rate vs golden labels (`rating_*` columns, normalised to lowercase).
6. **Stop** if every field's rate ≤ 30 %, or after 5 rounds.
7. Otherwise collect mismatch rows → fill `OptPrompt` → POST to webhook → use response as next prompt.

In [6]:
os.makedirs(RESULT_DIR, exist_ok=True)

current_prompt = prompt_0
all_rounds: list[dict] = []

for round_idx in range(MAX_ROUNDS):
    round_name  = f"prompt_{round_idx}"
    out_path    = os.path.join(RESULT_DIR, f"test_result_{round_name}.xlsx")
    prompt_path = os.path.join(RESULT_DIR, f"{round_name}.txt")

    print(f"\n{'=' * 60}")
    print(f"Round {round_idx}  |  {round_name}")
    print(f"{'=' * 60}")

    # ── Save prompt as text file ───────────────────────────────────────────
    with open(prompt_path, "w", encoding="utf-8") as fh:
        fh.write(current_prompt)
    print(f"  Saved prompt → {prompt_path}")

    # ── Check if result xlsx already exists — skip evaluation if so ───────
    if os.path.exists(out_path):
        print(f"  Result already exists → {out_path}  (skipping evaluation)")
        df_test_result = pd.read_excel(out_path)
        df_pred = pd.DataFrame({f: df_test_result[f"pred_{f}"].values for f in SCORE_FIELDS})
    else:
        # ── 1. Run evaluation ──────────────────────────────────────────────
        predicted: list[dict] = []
        errors:    list[str | None] = []

        for _, row in tqdm(df_golden.iterrows(), total=len(df_golden), desc=round_name):
            try:
                filled = fill_prompt(current_prompt, row)
                raw    = call_webhook(filled)
                parsed = parse_json_response(raw)
                predicted.append({f: parsed.get(f) for f in SCORE_FIELDS})
                errors.append(None)
            except Exception as exc:
                predicted.append({f: None for f in SCORE_FIELDS})
                errors.append(str(exc))
            time.sleep(DELAY_BETWEEN)

        df_pred = pd.DataFrame(predicted)

        # ── 2. Build test_result dataframe ────────────────────────────────
        df_test_result = df_golden[["testId", "userMessage", "expected_route"]].copy()
        for f in SCORE_FIELDS:
            df_test_result[golden_col(f)]  = df_golden[golden_col(f)].values
            df_test_result[f"pred_{f}"]    = df_pred[f].values
            df_test_result[f"match_{f}"]   = (df_pred[f].values == df_golden[golden_col(f)].values)
        df_test_result["error"] = errors

        df_test_result.to_excel(out_path, index=False)
        print(f"  Saved → {out_path}")

    # ── 3. Compute per-field mismatch rates ───────────────────────────────
    mismatch_rates: dict[str, float] = {}
    for f in SCORE_FIELDS:
        valid = df_golden[golden_col(f)].notna() & df_pred[f].notna()
        if valid.sum():
            mismatch_rates[f] = float((~df_test_result[f"match_{f}"][valid]).mean())
        else:
            mismatch_rates[f] = float("nan")

    print("\n  Mismatch rates:")
    for f, rate in mismatch_rates.items():
        tag = "✓" if (not pd.isna(rate) and rate <= MISMATCH_THRESHOLD) else "✗"
        print(f"    {f:25s}: {rate:.1%}  {tag}")

    all_rounds.append({
        "round":          round_idx,
        "prompt_name":    round_name,
        "prompt":         current_prompt,
        "df_test_result": df_test_result.copy(),
        "mismatch_rates": mismatch_rates.copy(),
    })

    # ── 4. Stopping condition ─────────────────────────────────────────────
    valid_rates  = [v for v in mismatch_rates.values() if not pd.isna(v)]
    max_mismatch = max(valid_rates) if valid_rates else float("nan")

    if not pd.isna(max_mismatch) and max_mismatch <= MISMATCH_THRESHOLD:
        print(f"\n  All fields within {MISMATCH_THRESHOLD:.0%} threshold. Stopping.")
        break

    if round_idx == MAX_ROUNDS - 1:
        print(f"\n  Reached maximum {MAX_ROUNDS} rounds. Stopping.")
        break

    # ── 5. Collect mismatched rows ─────────────────────────────────────────
    mismatch_mask = pd.Series(False, index=df_golden.index)
    for f in SCORE_FIELDS:
        mismatch_mask |= (df_pred[f].values != df_golden[golden_col(f)].values)

    context_cols      = [
        "testId", "userMessage", "LastAImessage", "PrevAIagent",
        "narrative", "expected_route", "reason", "result_narrative",
    ]
    golden_label_cols = [golden_col(f) for f in SCORE_FIELDS]

    df_mismatch = (
        df_golden[mismatch_mask][context_cols + golden_label_cols]
        .copy()
        .reset_index(drop=True)
    )
    df_pred_mis = df_pred[mismatch_mask.values].reset_index(drop=True)
    for f in SCORE_FIELDS:
        df_mismatch[f"pred_{f}"] = df_pred_mis[f].values

    mismatch_json = df_mismatch.to_json(orient="records", force_ascii=False, indent=2)
    print(f"\n  {mismatch_mask.sum()} mismatched rows → OptPrompt.")

    # ── 6. Get improved prompt ─────────────────────────────────────────────
    opt_filled = fill_opt_prompt(OptPrompt, current_prompt, mismatch_json)
    print("  Requesting improved prompt…")
    opt_resp       = call_webhook(opt_filled)
    current_prompt = extract_prompt_text(opt_resp)
    print(f"  New prompt ({len(current_prompt)} chars) ready for round {round_idx + 1}.")

print(f"\nOptimisation complete — {len(all_rounds)} round(s) executed.")


Round 0  |  prompt_0
  Saved prompt → classification_test_result/prompt_0.txt


prompt_0:   0%|          | 0/30 [00:00<?, ?it/s]

  Saved → classification_test_result/test_result_prompt_0.xlsx

  Mismatch rates:
    narrative                : 70.0%  ✗

  21 mismatched rows → OptPrompt.
  Requesting improved prompt…
  New prompt (2674 chars) ready for round 1.

Round 1  |  prompt_1
  Saved prompt → classification_test_result/prompt_1.txt


prompt_1:   0%|          | 0/30 [00:00<?, ?it/s]

  Saved → classification_test_result/test_result_prompt_1.xlsx

  Mismatch rates:
    narrative                : 16.7%  ✓

  All fields within 30% threshold. Stopping.

Optimisation complete — 2 round(s) executed.


## Results Summary

In [7]:
summary_rows = []
for r in all_rounds:
    row = {"round": r["round"], "prompt_name": r["prompt_name"]}
    row.update({f: f'{v:.1%}' for f, v in r["mismatch_rates"].items()})
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print("=" * 60)
print("OPTIMISATION SUMMARY — mismatch rate per field")
print("=" * 60)
display(df_summary)

best = min(
    all_rounds,
    key=lambda r: max(
        (v for v in r["mismatch_rates"].values() if not pd.isna(v)),
        default=float("inf"),
    ),
)
print(f"\nBest round : {best['prompt_name']}")
print(
    f"Max mismatch: "
    f"{max(v for v in best['mismatch_rates'].values() if not pd.isna(v)):.1%}"
)
print("\nAccess results:")
print("  all_rounds[N]['df_test_result']  — labelled DataFrame for round N")
print("  all_rounds[N]['prompt']          — prompt text used in round N")
print("  all_rounds[-1]['prompt']         — final (best last) prompt")

OPTIMISATION SUMMARY — mismatch rate per field


,round,prompt_name,narrative
0,0,prompt_0,70.0%
1,1,prompt_1,16.7%



Best round : prompt_1
Max mismatch: 16.7%

Access results:
  all_rounds[N]['df_test_result']  — labelled DataFrame for round N
  all_rounds[N]['prompt']          — prompt text used in round N
  all_rounds[-1]['prompt']         — final (best last) prompt
